In [3]:
import os
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings


# 현재 노트북 위치
BASE_DIR = Path(
    r"C:\Users\Playdata\Desktop\mle-01-p2-team2\홍기표"
)
# .env 불러오기
load_dotenv(BASE_DIR / ".env")
load_dotenv(BASE_DIR.parent / ".env")

print("현재 위치:", BASE_DIR)
print("API KEY 존재:", bool(os.getenv("OPENAI_API_KEY")))

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY를 찾지 못했습니다.")

현재 위치: C:\Users\Playdata\Desktop\mle-01-p2-team2\홍기표
API KEY 존재: True


In [4]:
file_path = BASE_DIR / "output" / "recipes_classified_cleaned.jsonl"

df = pd.read_json(
    file_path,
    lines=True,
    encoding="utf-8"
)

print("데이터 개수:", len(df))
print("컬럼:", df.columns.tolist())

데이터 개수: 9381
컬럼: ['source', 'source_id', 'source_url', 'title', 'description', 'servings', 'cooking_time', 'difficulty', 'ingredients', 'seasonings', 'tools', 'steps', 'created_at', 'updated_at', 'views', 'selection_reason', 'recipe_uid', 'ingredients_clean', 'seasonings_clean', 'tools_clean', 'is_collection', 'graph_eligible', 'preprocess_warnings', 'document_text', 'servings_raw', 'servings_min', 'servings_max', 'cooking_time_raw', 'minutes_min', 'minutes_max', 'dish_group', 'dish_type', 'cooking_method']


In [5]:
EMBED_MODEL = "text-embedding-3-large"
EMBED_DIM = 768

embedder = OpenAIEmbeddings(
    model=EMBED_MODEL,
    dimensions=EMBED_DIM
)

print("임베딩 모델 준비 완료")

임베딩 모델 준비 완료


In [6]:
embedding_texts = []
metadatas = []


def join_normalized_names(items):
    if not isinstance(items, list):
        return ""

    names = []

    for item in items:
        if isinstance(item, dict):
            name = item.get("name_normalized")

            if name:
                names.append(str(name))

        else:
            names.append(str(item))

    return ", ".join(names)


def clean_metadata(metadata):
    cleaned = {}

    for key, value in metadata.items():

        # None 제거
        if value is None:
            continue

        # numpy 타입 -> Python 타입
        if isinstance(value, np.generic):
            value = value.item()

        # NaN 제거
        if isinstance(value, float) and pd.isna(value):
            continue

        # 허용 타입만 저장
        if isinstance(value, (str, int, float, bool)):
            cleaned[key] = value

        else:
            cleaned[key] = str(value)

    return cleaned


for _, recipe in df.iterrows():

    ingredients = join_normalized_names(
        recipe["ingredients_clean"]
    )

    seasonings = join_normalized_names(
        recipe["seasonings_clean"]
    )

    embedding_text = f"""
요리명: {recipe['title']}
음식 분류: {recipe['dish_group']}
음식 종류: {recipe['dish_type']}
설명: {recipe['description']}
재료: {ingredients}
양념: {seasonings}
조리법: {recipe['cooking_method']}
""".strip()

    embedding_texts.append(embedding_text)

    metadata = {
        "recipe_uid": recipe["recipe_uid"],
        "title": recipe["title"],
        "dish_group": recipe["dish_group"],
        "dish_type": recipe["dish_type"],
        "difficulty": recipe["difficulty"],
        "servings_min": recipe["servings_min"],
        "servings_max": recipe["servings_max"],
        "minutes_min": recipe["minutes_min"],
        "minutes_max": recipe["minutes_max"],
        "views": recipe["views"],
        "source_url": recipe["source_url"],
    }

    metadatas.append(
        clean_metadata(metadata)
    )

In [7]:
vectors = []

batch_size = 100

for i in range(0, len(embedding_texts), batch_size):

    batch = embedding_texts[
        i:i + batch_size
    ]

    batch_vectors = embedder.embed_documents(
        batch
    )

    vectors.extend(batch_vectors)

    end = min(
        i + batch_size,
        len(embedding_texts)
    )

    print(
        f"{end} / {len(embedding_texts)} 임베딩 완료"
    )

RateLimitError: Error code: 429 - {'error': {'message': 'Your project has reached its configured enforced spend limit. Update your limit at https://platform.openai.com/settings/proj_AMl1WtY0x3TbbRw5Tm3Ok9ZJ/limits.', 'type': 'insufficient_quota', 'param': None, 'code': 'project_spend_limit_exceeded'}}

In [ ]:
backup_path = BASE_DIR / "recipe_vectors_backup.pkl"

backup = {
    "documents": embedding_texts,
    "metadatas": metadatas,
    "embeddings": vectors
}

with open(backup_path, "wb") as f:
    pickle.dump(backup, f)

print("백업 완료:", backup_path)
print("파일 존재:", backup_path.exists())

백업 완료: C:\Users\Playdata\Desktop\mle-01-p2-team2\홍기표\recipe_vectors_backup.pkl
파일 존재: True


In [ ]:
ids = [
    str(metadata["recipe_uid"])
    for metadata in metadatas
]

print("ID 개수:", len(ids))
print(ids[:5])

assert len(ids) == len(set(ids))

print("ID 중복 없음")

ID 개수: 9381
['10000recipe_6876357', '10000recipe_1785098', '10000recipe_6873683', '10000recipe_6903507', '10000recipe_6879215']
ID 중복 없음


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

ENV_PATH = Path(
    r"C:\Users\Playdata\Desktop\mle-01-p2-team2\홍기표\.env"
)

load_dotenv(ENV_PATH, override=True)

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USER")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

print("URI:", NEO4J_URI)
print("USER:", NEO4J_USER)
print("DATABASE:", NEO4J_DATABASE)
print("비밀번호 설정됨:", bool(NEO4J_PASSWORD))

URI: neo4j+ssc://11769529.databases.neo4j.io
USER: 11769529
DATABASE: neo4j
비밀번호 설정됨: True


In [ ]:
from pathlib import Path
from dotenv import dotenv_values
from neo4j import GraphDatabase

env_path = Path(
    r"C:\Users\Playdata\Desktop\mle-01-p2-team2\홍기표\.env"
)

config = dotenv_values(env_path)

uri = config["NEO4J_URI"].strip()
user = config["NEO4J_USER"].strip()
password = config["NEO4J_PASSWORD"]
database = config["NEO4J_DATABASE"].strip()

print("URI:", repr(uri))
print("USER:", repr(user))
print("DATABASE:", repr(database))
print("비밀번호 설정됨:", bool(password))

driver = GraphDatabase.driver(
    uri,
    auth=(user, password),
)

try:
    driver.verify_connectivity()
    print("연결 성공")
finally:
    driver.close()

URI: 'neo4j+ssc://11769529.databases.neo4j.io'
USER: '11769529'
DATABASE: 'neo4j'
비밀번호 설정됨: True
연결 성공


In [ ]:


driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD),
)
driver.verify_connectivity()

embedding_rows = [
    {
        "recipe_uid": metadata["recipe_uid"],
        "embedding": vector,
    }
    for metadata, vector in zip(metadatas, vectors)
]

assert len(embedding_rows) == len(ids)

batch_size = 200
with driver.session(database=NEO4J_DATABASE) as session:
    for start in range(0, len(embedding_rows), batch_size):
        batch = embedding_rows[start:start + batch_size]
        matched = session.run(
            """
            UNWIND $rows AS row
            MATCH (d:Dish {recipe_uid: row.recipe_uid})
            SET d.embedding = row.embedding
            RETURN count(d) AS matched
            """,
            rows=batch,
        ).single()["matched"]
        print(f"{min(start + batch_size, len(embedding_rows))} / {len(embedding_rows)}개 임베딩 적재, 매칭 노드: {matched}")

print("Neo4j 임베딩 적재 완료")
driver.verify_connectivity()

with driver.session(database="neo4j") as session:
    result = session.run("RETURN 1 AS ok").single()
    print(result["ok"])

AuthError: {neo4j_code: Neo.ClientError.Security.Unauthorized} {message: The client is unauthorized due to authentication failure.} {gql_status: 42NFF} {gql_status_description: error: syntax error or access rule violation - permission/access denied. Access denied, see the security logs for details.}

In [ ]:
print("URI:", NEO4J_URI)
print("USER:", NEO4J_USER)
print("DATABASE:", NEO4J_DATABASE)

URI: neo4j+ssc://11769529.databases.neo4j.io
USER: neo4j
DATABASE: neo4j


In [ ]:
with driver.session(database=NEO4J_DATABASE) as session:
    session.run(
        """
        CREATE VECTOR INDEX dish_embedding_index IF NOT EXISTS
        FOR (d:Dish) ON (d.embedding)
        OPTIONS {
            indexConfig: {
                `vector.dimensions`: 768,
                `vector.similarity_function`: 'cosine'
            }
        }
        """
    ).consume()

    indexes = session.run("SHOW VECTOR INDEXES").data()

print("벡터 인덱스 생성/확인 완료")
for index in indexes:
    print(index)

AuthError: {neo4j_code: Neo.ClientError.Security.Unauthorized} {message: The client is unauthorized due to authentication failure.} {gql_status: 42NFF} {gql_status_description: error: syntax error or access rule violation - permission/access denied. Access denied, see the security logs for details.}